# ✈️ Agentic AI Travel Planner (LangChain)

Run the cells top to bottom. Steps: install → upload data → write modules → test tools (no key) → run the agent → launch the Streamlit app.

> This project uses a pre-trained LLM driving tools (agentic AI), so there is no model to train. The notebook covers the data tools, the agent, and the deployable app.

## 1. Install

In [ ]:
!pip install -q streamlit langchain-core langchain-openai langchain-google-genai requests

## 2. Upload the three datasets
Select `flights.json`, `hotels.json`, `places.json` when prompted.

In [ ]:
import os, shutil
from google.colab import files
os.makedirs('data', exist_ok=True)
for name in files.upload():
    shutil.move(name, f'data/{name}')
print(os.listdir('data'))

## 3. Write the project files

In [ ]:
%%writefile travel_tools.py
"""Core travel-planning logic: flights, hotels, places, weather and budget.

These are plain Python functions that return JSON-serialisable data.
`agent.py` wraps them as LangChain tools. Keeping them separate makes them
easy to test without an LLM or an API key.
"""
from __future__ import annotations

import json
import math
from datetime import date, datetime, timedelta
from functools import lru_cache
from pathlib import Path

import requests

DATA_DIR = Path(__file__).parent / "data"

# Latitude / longitude for every city present in the datasets (used for weather).
CITY_COORDS = {
    "delhi": (28.6139, 77.2090),
    "mumbai": (19.0760, 72.8777),
    "goa": (15.2993, 74.1240),
    "bangalore": (12.9716, 77.5946),
    "chennai": (13.0827, 80.2707),
    "hyderabad": (17.3850, 78.4867),
    "kolkata": (22.5726, 88.3639),
    "jaipur": (26.9124, 75.7873),
}
CITIES = [c.title() for c in CITY_COORDS]

# WMO weather codes -> readable text (Open-Meteo returns these).
WEATHER_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Rime fog", 51: "Light drizzle", 53: "Drizzle",
    55: "Dense drizzle", 61: "Light rain", 63: "Rain", 65: "Heavy rain",
    71: "Light snow", 73: "Snow", 75: "Heavy snow", 80: "Rain showers",
    81: "Heavy rain showers", 82: "Violent rain showers",
    95: "Thunderstorm", 96: "Thunderstorm with hail", 99: "Severe thunderstorm",
}


# --------------------------------------------------------------------------
# Data loading
# --------------------------------------------------------------------------
@lru_cache(maxsize=None)
def _load(name: str) -> list[dict]:
    """Load one of the JSON datasets from the data folder."""
    path = DATA_DIR / f"{name}.json"
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"Dataset not found: {path}") from exc


def _norm(text: str) -> str:
    return (text or "").strip().lower()


def available_place_types() -> list[str]:
    """All attraction types that exist in places.json."""
    return sorted({p["type"] for p in _load("places")})


# --------------------------------------------------------------------------
# Flights
# --------------------------------------------------------------------------
def _duration_hours(flight: dict) -> float:
    dep = datetime.fromisoformat(flight["departure_time"])
    arr = datetime.fromisoformat(flight["arrival_time"])
    return round((arr - dep).total_seconds() / 3600, 1)


def search_flights(origin: str, destination: str, travel_date: str = "",
                   sort_by: str = "cheapest", limit: int = 5) -> dict:
    """Find flights for a route.

    sort_by: "cheapest" or "fastest". If travel_date (YYYY-MM-DD) is given,
    flights on that exact date are ranked first, then the closest dates.
    """
    try:
        matches = [
            f for f in _load("flights")
            if _norm(f["from"]) == _norm(origin) and _norm(f["to"]) == _norm(destination)
        ]
        if not matches:
            routes = sorted({f"{f['from']} -> {f['to']}" for f in _load("flights")})
            return {"found": False,
                    "message": f"No flight from {origin} to {destination} in the dataset.",
                    "hint": "Try a connecting route via another city.",
                    "available_routes": routes}

        target = datetime.strptime(travel_date, "%Y-%m-%d").date() if travel_date else None
        results = []
        for f in matches:
            dep = datetime.fromisoformat(f["departure_time"])
            # Sample flights are all from 2025, so compare by time of year (ignore the year)
            # and treat Dec/Jan as neighbours.
            gap = 0
            if target:
                diff = abs(dep.timetuple().tm_yday - target.timetuple().tm_yday)
                gap = min(diff, 365 - diff)
            results.append({
                **f,
                "duration_hours": _duration_hours(f),
                "exact_date_match": bool(target and dep.date() == target),
                "days_from_requested_date_ignoring_year": gap if target else None,
            })

        key = (lambda r: r["duration_hours"]) if sort_by == "fastest" else (lambda r: r["price"])
        # With a date, closeness is primary; then price/duration.
        results.sort(key=lambda r: (r["days_from_requested_date_ignoring_year"] or 0, key(r)))
        return {"found": True, "count": len(results), "flights": results[:limit]}
    except ValueError:
        return {"found": False, "message": "travel_date must be in YYYY-MM-DD format."}
    except Exception as exc:  # noqa: BLE001 - tool must never crash the agent
        return {"found": False, "message": f"Flight search failed: {exc}"}


# --------------------------------------------------------------------------
# Hotels
# --------------------------------------------------------------------------
def recommend_hotels(city: str, min_stars: int = 1, max_price_per_night: float = 0,
                     amenities: list[str] | None = None, sort_by: str = "best_value",
                     limit: int = 5) -> dict:
    """Recommend hotels in a city.

    sort_by: "best_value" (stars per rupee), "cheapest" or "highest_rated".
    max_price_per_night = 0 means no price cap.
    """
    try:
        wanted = {_norm(a) for a in (amenities or [])}
        hotels = [
            h for h in _load("hotels")
            if _norm(h["city"]) == _norm(city)
            and h["stars"] >= min_stars
            and (not max_price_per_night or h["price_per_night"] <= max_price_per_night)
            and wanted.issubset({_norm(a) for a in h["amenities"]})
        ]
        if not hotels:
            return {"found": False, "message": f"No hotels in {city} match those filters."}

        if sort_by == "cheapest":
            hotels.sort(key=lambda h: h["price_per_night"])
        elif sort_by == "highest_rated":
            hotels.sort(key=lambda h: (-h["stars"], h["price_per_night"]))
        else:
            hotels.sort(key=lambda h: -h["stars"] / h["price_per_night"])
        return {"found": True, "count": len(hotels), "hotels": hotels[:limit]}
    except Exception as exc:  # noqa: BLE001
        return {"found": False, "message": f"Hotel search failed: {exc}"}


# --------------------------------------------------------------------------
# Places
# --------------------------------------------------------------------------
def discover_places(city: str, place_types: list[str] | None = None,
                    min_rating: float = 0, limit: int = 8) -> dict:
    """Recommend attractions in a city, best rated first."""
    try:
        types = {_norm(t) for t in (place_types or [])}
        places = [
            p for p in _load("places")
            if _norm(p["city"]) == _norm(city)
            and p["rating"] >= min_rating
            and (not types or _norm(p["type"]) in types)
        ]
        if not places:
            return {"found": False,
                    "message": f"No places in {city} match those filters.",
                    "available_types": available_place_types()}
        places.sort(key=lambda p: -p["rating"])
        return {"found": True, "count": len(places), "places": places[:limit]}
    except Exception as exc:  # noqa: BLE001
        return {"found": False, "message": f"Place search failed: {exc}"}


# --------------------------------------------------------------------------
# Weather (Open-Meteo, free, no API key)
# --------------------------------------------------------------------------
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
DAILY_VARS = "weather_code,temperature_2m_max,temperature_2m_min,precipitation_sum"


def get_weather(city: str, start_date: str, end_date: str) -> dict:
    """Daily weather for a city between two dates (YYYY-MM-DD).

    Open-Meteo forecasts cover ~16 days ahead. For dates further out we return
    what the weather was on the same dates one year earlier and say so.
    """
    try:
        coords = CITY_COORDS.get(_norm(city))
        if not coords:
            return {"found": False, "message": f"Unknown city '{city}'. Known: {CITIES}"}

        start = datetime.strptime(start_date, "%Y-%m-%d").date()
        end = datetime.strptime(end_date, "%Y-%m-%d").date()
        if end < start:
            return {"found": False, "message": "end_date is before start_date."}

        today = date.today()
        mode = "forecast"
        url = FORECAST_URL
        if start < today - timedelta(days=1) or end > today + timedelta(days=15):
            # Outside the forecast window -> use historical data for the same dates.
            url, mode = ARCHIVE_URL, "historical"
            while end >= today - timedelta(days=7):
                start -= timedelta(days=365)
                end -= timedelta(days=365)

        resp = requests.get(url, params={
            "latitude": coords[0], "longitude": coords[1], "daily": DAILY_VARS,
            "timezone": "auto", "start_date": start.isoformat(), "end_date": end.isoformat(),
        }, timeout=15)
        resp.raise_for_status()
        daily = resp.json()["daily"]

        days = []
        for i, d in enumerate(daily["time"]):
            days.append({
                "date": d,
                "condition": WEATHER_CODES.get(daily["weather_code"][i], "Unknown"),
                "max_temp_c": daily["temperature_2m_max"][i],
                "min_temp_c": daily["temperature_2m_min"][i],
                "rain_mm": daily["precipitation_sum"][i],
            })
        note = ("Live forecast." if mode == "forecast" else
                "Trip dates are outside the 16-day forecast window, so this shows the actual "
                f"weather on the same dates a year earlier ({days[0]['date']} onward) as a guide.")
        return {"found": True, "city": city, "source": mode, "note": note, "days": days}
    except requests.RequestException as exc:
        return {"found": False, "message": f"Weather API error: {exc}"}
    except Exception as exc:  # noqa: BLE001
        return {"found": False, "message": f"Weather lookup failed: {exc}"}


# --------------------------------------------------------------------------
# Budget
# --------------------------------------------------------------------------
def estimate_budget(flight_price: float, hotel_price_per_night: float, nights: int,
                    travelers: int = 1, daily_local_expense: float = 1500,
                    return_flight_price: float = 0) -> dict:
    """Total trip cost = flights + hotel rooms + food/local travel.

    Rooms are assumed to sleep two people. daily_local_expense is per person.
    """
    try:
        rooms = math.ceil(travelers / 2)
        flights_total = (flight_price + return_flight_price) * travelers
        hotel_total = hotel_price_per_night * nights * rooms
        local_total = daily_local_expense * (nights + 1) * travelers
        return {
            "travelers": travelers, "rooms": rooms, "nights": nights,
            "flight_total": round(flights_total),
            "hotel_total": round(hotel_total),
            "food_and_local_travel_total": round(local_total),
            "grand_total": round(flights_total + hotel_total + local_total),
            "currency": "INR",
        }
    except Exception as exc:  # noqa: BLE001
        return {"error": f"Budget estimation failed: {exc}"}


In [ ]:
%%writefile agent.py
"""LangChain tool-calling travel agent.

The LLM decides which tools to call (flights, hotels, places, weather, budget),
reads the results, and writes the final itinerary. Works with OpenAI or Google
Gemini - only the API key and model name change.
"""
from __future__ import annotations

import json
from typing import Callable, Optional

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

import travel_tools as tt

CITIES = tt.CITIES
PLACE_TYPES = tt.available_place_types()


# --------------------------------------------------------------------------
# Tools (thin wrappers - the docstrings are what the LLM reads)
# --------------------------------------------------------------------------
@tool
def flight_search_tool(origin: str, destination: str, travel_date: str = "",
                       sort_by: str = "cheapest") -> str:
    """Search flights from origin city to destination city. travel_date is YYYY-MM-DD
    (optional). sort_by is 'cheapest' or 'fastest'. Returns flights with price, times and
    duration. If no flight exists, returns the available routes so you can try a connection."""
    return json.dumps(tt.search_flights(origin, destination, travel_date, sort_by))


@tool
def hotel_recommendation_tool(city: str, min_stars: int = 1, max_price_per_night: float = 0,
                              sort_by: str = "best_value") -> str:
    """Recommend hotels in a city. min_stars 1-5. max_price_per_night in INR (0 = no cap).
    sort_by is 'best_value', 'cheapest' or 'highest_rated'."""
    return json.dumps(tt.recommend_hotels(city, min_stars, max_price_per_night,
                                          sort_by=sort_by))


@tool
def places_discovery_tool(city: str, place_types: Optional[list[str]] = None,
                          min_rating: float = 0, limit: int = 8) -> str:
    """Find attractions in a city, best rated first. place_types can include: beach, fort,
    lake, market, monument, museum, park, temple. Leave place_types empty for all types."""
    return json.dumps(tt.discover_places(city, place_types, min_rating, limit))


@tool
def weather_lookup_tool(city: str, start_date: str, end_date: str) -> str:
    """Get daily weather (condition, max/min temperature in C, rain in mm) for a city between
    start_date and end_date (YYYY-MM-DD). Uses the free Open-Meteo API."""
    return json.dumps(tt.get_weather(city, start_date, end_date))


@tool
def budget_estimation_tool(flight_price: float, hotel_price_per_night: float, nights: int,
                           travelers: int = 1, daily_local_expense: float = 1500,
                           return_flight_price: float = 0) -> str:
    """Estimate total trip cost in INR: flights + hotel + food/local travel.
    flight_price is per person one-way; add return_flight_price if a return flight is used.
    daily_local_expense is per person per day."""
    return json.dumps(tt.estimate_budget(flight_price, hotel_price_per_night, nights,
                                         travelers, daily_local_expense, return_flight_price))


TOOLS = [flight_search_tool, hotel_recommendation_tool, places_discovery_tool,
         weather_lookup_tool, budget_estimation_tool]
TOOL_MAP = {t.name: t for t in TOOLS}

SYSTEM_PROMPT = f"""You are an expert travel planning agent. Plan trips using ONLY the tools
provided; never invent flights, hotels or places.

Cities in the datasets: {", ".join(CITIES)}.

Workflow:
1. Search flights origin -> destination (cheapest unless the user says otherwise). Also search the
   return flight destination -> origin. If no direct route exists, try a connection through
   another city and explain it. If still impossible, say so honestly.
2. Recommend hotels in the destination that respect the user's budget and preference.
3. Find attractions matching the user's interests (use all types if none are given).
4. Get weather for the trip dates (start date to start date + days - 1).
5. Estimate the budget with the budget tool (nights = days - 1).
6. Write the final answer.

Rules:
- Flight data is sample data and its dates may not match the requested dates. If the dates differ,
  say so clearly and present it as the closest available option.
- If the user gave a total budget and the plan exceeds it, say so and suggest cheaper choices.
- Spread places across days (2-3 per day, best rated first, no repeats). If there are fewer places
  than needed, add free-time/rest suggestions instead of making up places.
- Explain WHY you chose each flight and hotel.

Final answer format (exactly):
## Your <N>-Day Trip to <Destination> (<start> - <end>)
**Flight:** ... **Hotel:** ... (one short line each, with price)
### Weather  - one line per day
### Itinerary - Day 1, Day 2 ... each with places and a short note
### Budget - flight, hotel, food & local travel, total
### Why these choices - 2-4 bullets
Then a fenced ```json block with this structure:
{{"trip_summary": {{"destination": "", "start_date": "", "end_date": "", "days": 0, "travelers": 0}},
 "flight": {{}}, "return_flight": {{}}, "hotel": {{}},
 "weather": [{{"date": "", "condition": "", "max_temp_c": 0}}],
 "itinerary": [{{"day": 1, "date": "", "places": [], "notes": ""}}],
 "budget": {{"flight": 0, "hotel": 0, "food_and_local": 0, "total": 0}},
 "reasoning": ""}}
"""


# --------------------------------------------------------------------------
# LLM factory
# --------------------------------------------------------------------------
DEFAULT_MODELS = {"OpenAI": "gpt-4o-mini", "Google Gemini": "gemini-2.5-flash"}


def build_llm(provider: str, api_key: str, model: str = "", temperature: float = 0.2):
    """Create a chat model for the chosen provider."""
    model = model or DEFAULT_MODELS[provider]
    if provider == "OpenAI":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model, api_key=api_key, temperature=temperature)
    if provider == "Google Gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model, google_api_key=api_key,
                                      temperature=temperature)
    raise ValueError(f"Unsupported provider: {provider}")


def _text(content) -> str:
    """Message content can be a string or a list of parts (Gemini) - normalise to text."""
    if isinstance(content, str):
        return content
    parts = []
    for part in content or []:
        parts.append(part if isinstance(part, str) else part.get("text", ""))
    return "".join(parts)


# --------------------------------------------------------------------------
# Agent loop
# --------------------------------------------------------------------------
def run_agent(llm, user_query: str, max_steps: int = 12,
              on_step: Optional[Callable[[dict], None]] = None) -> dict:
    """Run the ReAct-style tool-calling loop.

    Returns {"answer": str, "trace": [ {tool, args, result} ... ]}.
    on_step is called after every tool call (used by Streamlit to show progress).
    """
    llm_with_tools = llm.bind_tools(TOOLS)
    messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=user_query)]
    trace: list[dict] = []

    for _ in range(max_steps):
        ai = llm_with_tools.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return {"answer": _text(ai.content), "trace": trace}

        for call in ai.tool_calls:
            try:
                result = TOOL_MAP[call["name"]].invoke(call["args"])
            except Exception as exc:  # noqa: BLE001 - let the LLM see the error and recover
                result = json.dumps({"error": f"{type(exc).__name__}: {exc}"})
            step = {"tool": call["name"], "args": call["args"], "result": result}
            trace.append(step)
            if on_step:
                on_step(step)
            messages.append(ToolMessage(content=result, tool_call_id=call["id"],
                                        name=call["name"]))

    return {"answer": "The agent reached its step limit before finishing. Try again or "
                      "simplify the request.", "trace": trace}


def split_answer(answer: str) -> tuple[str, Optional[dict]]:
    """Separate the human-readable itinerary from the trailing JSON block."""
    import re
    match = re.search(r"```json\s*(\{.*?\})\s*```", answer, re.DOTALL)
    if not match:
        return answer, None
    try:
        data = json.loads(match.group(1))
    except json.JSONDecodeError:
        return answer, None
    return answer[:match.start()].rstrip(), data


In [ ]:
%%writefile app.py
"""Streamlit front-end for the Agentic AI Travel Planner."""
import json
import os
from datetime import date, timedelta

import streamlit as st

from agent import CITIES, DEFAULT_MODELS, PLACE_TYPES, build_llm, run_agent, split_answer

st.set_page_config(page_title="AI Travel Planner", page_icon="✈️", layout="wide")


def secret(name: str) -> str:
    """Read a key from Streamlit secrets or environment variables (never hard-code keys)."""
    try:
        return st.secrets.get(name, "") or os.environ.get(name, "")
    except Exception:  # no secrets file present
        return os.environ.get(name, "")


# ----------------------------- Sidebar: LLM settings -----------------------------
with st.sidebar:
    st.header("🔑 LLM settings")
    provider = st.selectbox("Provider", list(DEFAULT_MODELS))
    key_name = "OPENAI_API_KEY" if provider == "OpenAI" else "GOOGLE_API_KEY"
    api_key = st.text_input(f"{key_name}", value=secret(key_name), type="password",
                            help="Stored only in this session. On Streamlit Cloud, put it in Secrets.")
    model = st.text_input("Model", value=DEFAULT_MODELS[provider])
    st.caption("Weather comes from Open-Meteo (free, no key needed).")

# ----------------------------- Main: trip form -----------------------------
st.title("✈️ Agentic AI Travel Planner")
st.caption("A LangChain agent that picks flights, hotels and attractions, checks the weather "
           "and builds a budgeted day-wise itinerary.")

col1, col2, col3 = st.columns(3)
origin = col1.selectbox("From", CITIES, index=CITIES.index("Delhi"))
destination = col2.selectbox("To", CITIES, index=CITIES.index("Goa"))
start_date = col3.date_input("Start date", value=date.today() + timedelta(days=14))

col4, col5, col6 = st.columns(3)
days = col4.slider("Trip length (days)", 3, 7, 3)
travelers = col5.number_input("Travelers", 1, 10, 1)
budget = col6.number_input("Total budget (₹, 0 = no limit)", 0, 1_000_000, 0, step=1000)

col7, col8 = st.columns([2, 1])
interests = col7.multiselect("Interests", PLACE_TYPES, default=[])
hotel_pref = col8.selectbox("Hotel preference", ["Best value", "Cheapest", "Highest rated"])
notes = st.text_area("Anything else? (optional)", placeholder="e.g. prefer morning flights")

if st.button("Plan my trip", type="primary", use_container_width=True):
    if not api_key:
        st.error(f"Please enter your {key_name} in the sidebar.")
    elif origin == destination:
        st.error("Origin and destination must be different.")
    else:
        end_date = start_date + timedelta(days=days - 1)
        query = (
            f"Plan a {days}-day trip from {origin} to {destination}, "
            f"{start_date} to {end_date}, for {travelers} traveler(s). "
            f"Hotel preference: {hotel_pref}. "
            f"Interests: {', '.join(interests) if interests else 'no preference'}. "
            f"Total budget: {'₹' + str(budget) if budget else 'no limit'}. {notes}"
        )
        try:
            llm = build_llm(provider, api_key, model)
            with st.status("Agent is planning your trip...", expanded=True) as status:
                def show_step(step):
                    st.write(f"🔧 **{step['tool']}** — {json.dumps(step['args'])[:140]}")

                result = run_agent(llm, query, on_step=show_step)
                status.update(label="Plan ready!", state="complete", expanded=False)

            text, data = split_answer(result["answer"])
            tab1, tab2, tab3 = st.tabs(["📋 Itinerary", "🧾 JSON", "🧠 Agent reasoning"])
            with tab1:
                st.markdown(text)
            with tab2:
                if data:
                    st.json(data)
                    st.download_button("Download JSON", json.dumps(data, indent=2),
                                       "itinerary.json", "application/json")
                else:
                    st.info("The agent did not return a JSON block this time.")
            with tab3:
                for i, step in enumerate(result["trace"], 1):
                    with st.expander(f"{i}. {step['tool']}"):
                        st.write("**Input**")
                        st.json(step["args"])
                        st.write("**Output**")
                        st.code(step["result"], language="json")
        except ImportError as exc:
            st.error(f"Missing package: {exc}. Run `pip install -r requirements.txt`.")
        except Exception as exc:  # noqa: BLE001 - show API/auth errors in the UI
            st.error(f"Something went wrong: {exc}")


## 4. Test the tools (no API key needed)

In [ ]:
import json, travel_tools as tt
print(json.dumps(tt.search_flights('Goa', 'Bangalore', '2025-05-25'), indent=2)[:600])
print(json.dumps(tt.recommend_hotels('Goa', min_stars=4), indent=2)[:600])
print(json.dumps(tt.discover_places('Goa'), indent=2)[:600])
print(tt.get_weather('Goa', '2026-12-01', '2026-12-03'))
print(tt.estimate_budget(4800, 3200, 2, travelers=1, daily_local_expense=833))

## 5. Run the agent
Choose a provider and paste a key when prompted (OpenAI key, or a free Google AI Studio key for Gemini).

In [ ]:
from getpass import getpass
from agent import build_llm, run_agent, split_answer

PROVIDER = 'Google Gemini'   # or 'OpenAI'
llm = build_llm(PROVIDER, getpass('API key: '))

query = ('Plan a 3-day trip from Bangalore to Goa, 2026-12-10 to 2026-12-12, '
         'for 2 travelers. Hotel preference: Best value. Interests: beach, fort. No budget limit.')
result = run_agent(llm, query, on_step=lambda s: print('🔧', s['tool'], s['args']))
text, data = split_answer(result['answer'])
print(text)

In [ ]:
print(json.dumps(data, indent=2))

## 6. Launch the Streamlit app from Colab
This exposes the app through a temporary public URL (for demos). For a permanent deployment, push the files to GitHub and use Streamlit Community Cloud (see README).

In [ ]:
!npm install -g localtunnel > /dev/null 2>&1
import urllib.request, subprocess, time
print('Tunnel password (your Colab IP):', urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip())
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'])
time.sleep(5)
!npx localtunnel --port 8501